In [25]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from pathlib import Path
import time
load_dotenv()

True

In [ ]:
class GPTClient:
    def __init__(self):
        self.client = OpenAI()
        self.comment_list = self.read_json()

    def generate(self, prompt):
        response = self.client.responses.create(
            model="gpt-5.6-terra",
            input=prompt,
            reasoning={"effort": "high"} 
        )
        return response.output_text

    def main_pipeline(self):
        self.call_api(0)
    
    def read_json(self, filepath = "C:/Python Projects/knowledge-graph/json-exports/bug_comments_v1.json"):
        filepath = Path(filepath)
        if filepath.is_file():
            try:
                with open(filepath, "r", encoding='utf-8') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                return []

    def write_json(self, obj, filepath = None):
        filepath = Path(filepath)
        with open(filepath, "w", encoding='utf-8') as f:
            json.dump(obj, f, indent = 4)

    def call_api(self, start, filepath = "C:/Python Projects/knowledge-graph/json-exports/comments_analyzed_v1.json"):

        end = len(self.comment_list)
        analyzed_comm_list = self.read_json(filepath)
        start_time = time.time()

        for i in range(start, end):
            print("Processing Index: ", i)
            comment = self.comment_list[i]
            prompt = self.get_prompt(comment)
            response = self.generate(prompt)
            actual_dict = json.loads(response)
            analyzed_comm_list.append(actual_dict)
            if i%20 == 0:
                self.write_json(analyzed_comm_list, filepath)
                print("Time Taken: ", time.time() - start_time)
        print("Time Taken: ", time.time()-start_time)
            

    def get_prompt(self, comment):
        return f'''
        You are a Senior Enterprise Support Engineer responsible for converting Jira incident discussions into reusable organizational knowledge
        Your task is to analyze the complete chronological comment timeline of a single Jira incident and generate a concise, factual incident analysis.
        The comments represent the complete engineering discussion for the incident. Earlier comments may contain acknowledgements, intermediate investigations, incorrect assumptions or failed hypotheses, while later comments often contain the actual findings and final resolution.
        If both the original and English translation of a comment are available, always use the English translation. Otherwise, use the original comment.
        Your goal is NOT to summarize every comment.
        Your goal is to extract only the engineering knowledge that will help future engineers understand similar incidents.
 
        Instructions:

        - Read every comment in chronological order.
        - Ignore greetings, acknowledgements, signatures, polite language and repetitive information.
        - Do not invent or infer facts that are not supported by the comments.
        - If the root cause or resolution cannot be confidently determined, return null.
        - Preserve exact product names, table names, workflows, environments, systems, error messages and business objects whenever they appear.
        - Keep every field concise and factual.
        - Return ONLY valid JSON.
        - Do not include markdown.
        - Do not include explanations.

        Return the following JSON schema exactly:

        {{
            "issue_key": "",
            "investigation_summary": "",
            "important_findings": [
                "",
                "",
                ""
            ],
            "root_cause": "",
            "resolution": "",
            "technical_entities": [
                ""
            ]
        }}

        Field Guidelines

        issue_key
        - Copy directly from the input.

        investigation_summary
        - 2–4 sentences.
        - Explain the incident, investigation outcome and overall resolution.
        - Do not include greetings or unnecessary chronology.

        important_findings
        - 3–6 concise technical findings.
        - Each finding should represent an important engineering observation.
        - Remove duplicates.
        - Preserve chronological logic where appropriate.

        root_cause
        - The primary technical cause of the incident.
        - Return null if the comments do not clearly establish one.

        resolution
        - The final corrective action that resolved the incident.
        - Return null if unresolved.

        technical_entities
        - Extract every important technical entity explicitly mentioned.
        - Include products, systems, environments, tables, workflows, databases, interfaces, APIs, business objects, modules, technologies and error identifiers.
        - Do not infer entities that are not explicitly present.
        - Return an empty array if none are found.

        Input:

        {comment}
        '''



In [61]:
gpt_client = GPTClient()
gpt_client.main_pipeline()

Processing Index:  0


TypeError: the JSON object must be str, bytes or bytearray, not dict